In [ ]:
import tkinter as tk
from tkinter import ttk, simpledialog, messagebox
import folium
from folium.plugins import HeatMap
import webbrowser

district_locations = {
    # same district_locations dictionary as you provided earlier
    "Dehradun": [
        ("Dehradun", 30.3165, 78.0322),
        ("Rishikesh", 30.0869, 78.2676),
        ("Vikasnagar", 30.4571, 77.7725),
        ("Chakrata", 30.7038, 77.8628),
        ("Herbertpur", 30.4298, 77.7286),
        ("Doiwala", 30.1766, 78.1803)
    ],
     "Haridwar": [
        ("Haridwar", 29.9457, 78.1642),
        ("Roorkee", 29.8543, 77.8880),
        ("Laksar", 29.7614, 78.0421),
        ("Bahadrabad", 29.8510, 78.1166)
    ],
    "Nainital": [
        ("Nainital", 29.3803, 79.4636),
        ("Haldwani", 29.2190, 79.5231),
        ("Ramnagar", 29.3920, 79.1294),
        ("Kaladhungi", 29.3252, 79.3470),
        ("Bhimtal", 29.3443, 79.5678)
    ],
    "Udham Singh Nagar": [
        ("Rudrapur", 28.9874, 79.4144),
        ("Kashipur", 29.2130, 78.9577),
        ("Khatima", 28.9216, 79.9722),
        ("Sitarganj", 28.9326, 79.6985),
        ("Jaspur", 29.2785, 78.8276)
    ],
    "Pauri Garhwal": [
        ("Pauri", 30.1536, 78.7850),
        ("Kotdwar", 29.7443, 78.5222),
        ("Srinagar", 30.2217, 78.7802),
        ("Lansdowne", 29.8400, 78.6868),
        ("Satpuli", 30.1200, 78.9270)
    ],
    "Chamoli": [
        ("Gopeshwar", 30.4100, 79.3200),
        ("Joshimath", 30.5567, 79.5644),
        ("Karnaprayag", 30.2610, 79.2170),
        ("Nandaprayag", 30.3325, 79.3216),
        ("Chamoli", 30.4220, 79.2131)
    ],
    "Rudraprayag": [
        ("Rudraprayag", 30.2831, 78.9806),
        ("Agastyamuni", 30.4364, 79.0395),
        ("Ukhimath", 30.5037, 79.0570),
        ("Jakholi", 30.4107, 79.1270)
    ],
    "Tehri Garhwal": [
        ("New Tehri", 30.3794, 78.4803),
        ("Narendranagar", 30.1754, 78.2744),
        ("Chamba", 30.7405, 78.7775),
        ("Ghansali", 30.4750, 78.7310)
    ],
    "Almora": [
        ("Almora", 29.5892, 79.6460),
        ("Ranikhet", 29.6420, 79.4321),
        ("Dwarahat", 29.7755, 79.4273),
        ("Jainti", 29.5974, 79.5832)
    ],
    "Bageshwar": [
        ("Bageshwar", 29.8370, 79.7708),
        ("Kanda", 29.8592, 79.9115),
        ("Kapkot", 29.9112, 79.8494)
    ],
     "Pithoragarh": [
        ("Pithoragarh", 29.5803, 80.2181),
        ("Didihat", 29.7378, 80.2653),
        ("Berinag", 29.7915, 79.8584),
        ("Dharchula", 29.8816, 80.5357),
        ("Munsyari", 30.0696, 80.2390)
    ],
    "Champawat": [
        ("Champawat", 29.3352, 80.1025),
        ("Lohaghat", 29.4076, 80.0897),
        ("Tanakpur", 29.0727, 80.1110),
        ("Purnagiri", 29.0800, 80.1250)
    ]
}

severity_data_per_district = {}

def update_city_dropdown(selected_district):
    """
    Update city dropdown with city names and severity for the selected district.
    """
    city_dropdown['values'] = []  # Clear old values
    if not selected_district or selected_district not in district_locations:
        city_var.set('')
        city_dropdown['values'] = []
        city_dropdown.config(state='disabled')
        return
    
    district_data = severity_data_per_district.get(selected_district, {})
    city_list = []
    for place, lat, lon in district_locations[selected_district]:
        severity = district_data.get((lat, lon), 0.0)
        city_list.append(f"{place} (Severity: {severity:.2f})")
    
    city_dropdown['values'] = city_list
    city_dropdown.config(state='readonly')
    if city_list:
        city_var.set(city_list[0])
    else:
        city_var.set('')

def enter_severity_data(selected_district):
    """
    Prompts user to enter severity for each place in the selected district.
    Stores the values in the `severity_data_per_district`.
    Returns True if user entered at least one value; False if cancelled or none entered.
    """
    locations = district_locations.get(selected_district, [])
    if not locations:
        messagebox.showinfo("No Data", f"No predefined locations found for {selected_district}.")
        return False

    district_data = severity_data_per_district.get(selected_district, {})

    for place, lat, lon in locations:
        prev_value = district_data.get((lat, lon), 0.0)

        while True:
            value = simpledialog.askfloat(
                title="Severity Input",
                prompt=f"Enter severity for {place} (0.0 - 1.0):",
                minvalue=0.0,
                maxvalue=1.0,
                initialvalue=prev_value
            )
            if value is None:
                if messagebox.askyesno("Cancel Input", "Do you want to cancel severity input?"):
                    severity_data_per_district[selected_district] = district_data
                    update_city_dropdown(selected_district)  # update city dropdown to reflect current data
                    return len(district_data) > 0
                else:
                    continue
            else:
                district_data[(lat, lon)] = value
                break

    severity_data_per_district[selected_district] = district_data
    update_city_dropdown(selected_district)  # update city dropdown after all inputs
    return True

def show_summary(selected_district):
    district_data = severity_data_per_district.get(selected_district, {})
    if not district_data:
        return

    summary_lines = []
    for (lat, lon), severity in district_data.items():
        place_name = next((p for p, la, lo in district_locations[selected_district] if la == lat and lo == lon), None)
        summary_lines.append(f"{place_name}: {severity:.2f}")

    summary_text = "\n".join(summary_lines)
    messagebox.showinfo(f"Severity Summary for {selected_district}", summary_text)

def generate_heatmap(selected_district):
    if not selected_district:
        messagebox.showwarning("Selection Error", "Please select a district.")
        return

    proceed = enter_severity_data(selected_district)
    if not proceed:
        messagebox.showinfo("Operation Cancelled", "Severity data entry cancelled or no data entered.")
        return

    show_summary(selected_district)
    if not messagebox.askyesno("Generate Heatmap", f"Generate heatmap for {selected_district}?"):
        return

    district_data = severity_data_per_district.get(selected_district, {})
    if not district_data:
        messagebox.showinfo("No Data", "No severity data entered.")
        return

    avg_lat = sum(lat for lat, _ in district_data.keys()) / len(district_data)
    avg_lon = sum(lon for _, lon in district_data.keys()) / len(district_data)
    m = folium.Map(location=[avg_lat, avg_lon], zoom_start=10)

    heat_data = [[lat, lon, severity] for (lat, lon), severity in district_data.items()]
    HeatMap(heat_data, min_opacity=0.3, radius=25, blur=15, max_zoom=1).add_to(m)

    output_file = f"uttarakhand_{selected_district.lower().replace(' ', '_')}_heatmap.html"
    m.save(output_file)
    webbrowser.open(f"file://{output_file}")

def on_district_select(event=None):
    selected_district = district_var.get()
    if selected_district:
        generate_btn.config(state="normal")
        clear_btn.config(state="normal")
        update_city_dropdown(selected_district)
    else:
        generate_btn.config(state="disabled")
        clear_btn.config(state="disabled")
        city_dropdown.config(state="disabled")
        city_var.set('')

def clear_data():
    selected_district = district_var.get()
    if not selected_district:
        messagebox.showwarning("Selection Error", "Please select a district to clear data.")
        return
    if selected_district in severity_data_per_district:
        if messagebox.askyesno("Confirm Clear", f"Clear all severity data for {selected_district}?"):
            del severity_data_per_district[selected_district]
            messagebox.showinfo("Cleared", f"Severity data cleared for {selected_district}.")
            update_city_dropdown(selected_district)

root = tk.Tk()
root.title("Uttarakhand Disaster Severity Heatmap")
root.geometry("600x450")
root.resizable(False, False)

frame = ttk.Frame(root, padding=20)
frame.pack(expand=True, fill="both")

ttk.Label(frame, text="Select District", font=("Arial", 14)).grid(row=0, column=0, sticky="w", pady=(0, 5))

district_var = tk.StringVar()
district_dropdown = ttk.Combobox(frame, textvariable=district_var, state="readonly", font=("Arial", 12))
district_dropdown['values'] = list(district_locations.keys())
district_dropdown.grid(row=1, column=0, sticky="ew")
district_dropdown.bind("<<ComboboxSelected>>", on_district_select)

ttk.Label(frame, text="Select City with Severity", font=("Arial", 14)).grid(row=2, column=0, sticky="w", pady=(15, 5))

city_var = tk.StringVar()
city_dropdown = ttk.Combobox(frame, textvariable=city_var, state="disabled", font=("Arial", 12))
city_dropdown.grid(row=3, column=0, sticky="ew")

generate_btn = ttk.Button(frame, text="Generate Heatmap", command=lambda: generate_heatmap(district_var.get()), state="disabled")
generate_btn.grid(row=4, column=0, sticky="ew", pady=20)

clear_btn = ttk.Button(frame, text="Clear Severity Data", command=clear_data, state="disabled")
clear_btn.grid(row=5, column=0, sticky="ew")

instruction_text = (
    "Instructions:\n"
    "- Select a district.\n"
    "- Enter severity values (0.0 - 1.0) for each location.\n"
    "- View the heatmap.\n"
    "- Use 'Clear Severity Data' to reset entries for the district."
)
ttk.Label(frame, text=instruction_text, font=("Arial", 10), foreground="gray").grid(row=6, column=0, pady=10, sticky="w")

root.mainloop()
